Deteksi Dini Tingkat Stres Berdasarkan Pola Tidur Harian
## Coding Camp 2026 powered by DBS Foundation

**Tema:** Healthy Lives & Well-being  
**Data Scientist:** Izzah Huwaidah
**Fulstack Engineer:**
**AI Engineering:**   

---

## **Permasalahan & Solusi Utama**

### Problem Statement
Kesehatan mental, khususnya **stres**, merupakan isu global yang semakin meningkat terutama di kalangan mahasiswa dan pekerja. Banyak individu **tidak menyadari** bahwa pola tidur yang tidak teratur adalah salah satu sinyal peringatan dini stres kronis. Kurangnya *tools* deteksi mandiri berbasis data menyebabkan masyarakat baru mencari bantuan saat kondisi mental sudah kritis.

### Solusi yang Dikembangkan
Membangun **sistem deteksi dini tingkat stres** berbasis Machine Learning yang:
1. Menganalisis pola tidur harian (durasi, kualitas) sebagai fitur utama
2. Mempertimbangkan faktor gaya hidup (durasi   aktivitas fisik, BMI, riwayat kesehatan mental)
3. Mengklasifikasikan tingkat stres ke dalam 3 kategori: **Rendah, Sedang, Tinggi**
4. Memberikan rekomendasi personal untuk menjaga kesehatan mental

### Dampak yang Diharapkan
- Meningkatkan kesadaran masyarakat tentang hubungan pola tidur dan stres
- Menyediakan *early warning system* yang mudah diakses
- Mendorong deteksi mandiri sebelum kondisi mental memburuk

**Import Library**

## **1. Setup & Library Import**

In [4]:
# LIBRARY IMPORTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import math
from scipy import stats
from scipy.stats import skew, kurtosis
from scipy.stats import shapiro, chi2_contingency, kruskal, mannwhitneyu, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Plotting style (source :  AI)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='husl')
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860']

print('Libraries loaded - berhasil')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

Libraries loaded - berhasil
Pandas: 2.2.2 | NumPy: 2.0.2


**Load Dataset**

## **2. Data Gathering**

**Sumber Dataset:**
- **Dataset 1:**

Global Lifestyle and Lifespan Synthetic Dataset (Kaggle) - data gaya hidup global dengan fitur pola tidur, BMI, aktivitas fisik, dan tingkat stres.

- **Dataset 2:**

Mental Health Dataset (Kaggle) - data kesehatan mental dengan fitur riwayat mental, lingkungan kerja, skor depresi, kecemasan, dan risiko kesehatan mental.

**Strategi Penggabungan:** Kedua dataset memiliki kolom yang overlap (`sleep_hours`, `stress_level`, `physical_activity`) sehingga dapat digabungkan secara horizontal (concat) setelah harmonisasi nama kolom.

**Load Datasets**

In [15]:
# LOAD DATASETS 1 dan menampilkan 5 data pertama

# Dataset 1: Global Lifespan Determinant
url_1 = "https://raw.githubusercontent.com/IzzahHuwaidah/Predicting-Stress-Levels-Using-Sleep-Pattern-Data/refs/heads/main/Global_Lifespan_Determinant.csv"
df_lifestyle= pd.read_csv(url_1)
df_lifestyle.head()

,Unnamed: 0,TOTAL_AGE,SEX,SMOKING_STATUS,PHYSICAL_ACTIVITY_HOURS_PER_DAY,SLEEP_HOURS,BMI,STRESS_LEVEL,PROFESSION,EDUCATION_LEVEL,DIET_CALORIES,DISEASES_SUFFERIN_FROM,COUNTRY,HEALTH_RISK_SCORE,LIFESTYLE_SCORE
0,0,66.0,Male,Never,4.0,6.1,29.9,0.32,Teacher,PhD,3125,3,Canada,19.2863,7.8509
1,1,73.0,Female,Never,2.3,6.8,25.3,0.43,Doctor,Master's Degree,3462,11,Indonesia,23.9758,6.7363
2,2,71.0,Female,Never,5.9,7.9,24.8,0.69,Police,PhD,2021,9,Indonesia,36.6935,9.2592
3,3,66.8,Female,Medium,5.4,5.0,24.2,0.49,Soldier,Bachelor's Degree,3992,8,Canada,26.5986,3.4177
4,4,60.0,Female,High,3.9,8.9,26.1,0.74,Engineer,Some College,2463,7,Bangladesh,39.1394,2.9095


In [16]:
# load dataset 2 dan menampilkan 5 data pertama

# Dataset 2: Mental Health
url_2 = "https://raw.githubusercontent.com/IzzahHuwaidah/Predicting-Stress-Levels-Using-Sleep-Pattern-Data/refs/heads/main/mental_health_dataset.csv"
df_mental = pd.read_csv(url_2)
df_mental.head()

,age,gender,employment_status,work_environment,mental_health_history,seeks_treatment,stress_level,sleep_hours,physical_activity_days,depression_score,anxiety_score,social_support_score,productivity_score,mental_health_risk
0,56,Male,Employed,On-site,Yes,Yes,6,6.2,3,28,17,54,59.7,High
1,46,Female,Student,On-site,No,Yes,10,9.0,4,30,11,85,54.9,High
2,32,Female,Employed,On-site,Yes,No,7,7.7,2,24,7,62,61.3,Medium
3,60,Non-binary,Self-employed,On-site,No,No,4,4.5,4,6,0,95,97.0,Low
4,25,Female,Self-employed,On-site,Yes,Yes,3,5.4,0,24,12,70,69.0,High


## 3. Data Assessing

**Tujuan:** Menilai kondisi awal dataset untuk mengidentifikasi masalah data yang perlu ditangani sebelum analisis.

**Yang Diperiksa:**
- Missing values
- Duplicate rows
- Data types
- Inconsistent values / typos
- Outliers (statistik deskriptif)
- Range / distribusi nilai

In [17]:
# ASSESSING: Dataset 1
print('=' * 60)
print('ASSESSING: GLOBAL LIFESPAN DETERMINANT')
print('=' * 60)

# Ukuran Dataset 1
print(f'Ukuran Dataset: {df_lifestyle.shape}')

# Informasi Tipe Data
print('\n- Tipe Data:')
print(df_lifestyle.dtypes)

# Informasi Missing Values
print('\n- Missing Values:')
missing_1 = df_lifestyle.isnull().sum()
pct_1 = (missing_1 / len(df_lifestyle) * 100).round(2)
print(pd.DataFrame({'Missing': missing_1, 'Pct(%)': pct_1}))

# Informasi data duplikat
print(f'\n- Duplicate Rows: {df_lifestyle.duplicated().sum()}')

# Informasi statistik deskriptif untuk variabel numerik
print('\n- Statistik Deskriptif:')
df_lifestyle.describe()

ASSESSING: GLOBAL LIFESPAN DETERMINANT
Ukuran Dataset: (10000, 15)

- Tipe Data:
Unnamed: 0                           int64
TOTAL_AGE                          float64
SEX                                 object
SMOKING_STATUS                      object
PHYSICAL_ACTIVITY_HOURS_PER_DAY    float64
SLEEP_HOURS                        float64
BMI                                float64
STRESS_LEVEL                       float64
PROFESSION                          object
EDUCATION_LEVEL                     object
DIET_CALORIES                        int64
DISEASES_SUFFERIN_FROM               int64
COUNTRY                             object
HEALTH_RISK_SCORE                  float64
LIFESTYLE_SCORE                    float64
dtype: object

- Missing Values:
                                 Missing  Pct(%)
Unnamed: 0                             0     0.0
TOTAL_AGE                              0     0.0
SEX                                    0     0.0
SMOKING_STATUS                         0     

,Unnamed: 0,TOTAL_AGE,PHYSICAL_ACTIVITY_HOURS_PER_DAY,SLEEP_HOURS,BMI,STRESS_LEVEL,DIET_CALORIES,DISEASES_SUFFERIN_FROM,HEALTH_RISK_SCORE,LIFESTYLE_SCORE
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4999.50000,63.152305,5.392680,6.947610,25.447950,0.517192,2795.574600,7.564500,28.267125,6.682076
std,2886.89568,10.065661,2.982161,1.247396,3.858177,0.197072,589.896494,3.777018,9.950064,3.555209
min,0.00000,18.000000,0.500000,3.000000,18.000000,0.100000,1800.000000,2.000000,6.275900,2.620700
25%,2499.75000,59.000000,3.200000,6.100000,22.400000,0.370000,2342.000000,5.000000,21.019500,3.159425
50%,4999.50000,65.000000,4.800000,7.000000,25.100000,0.500000,2732.000000,7.000000,27.591100,6.492800
75%,7499.25000,70.000000,7.200000,8.000000,28.000000,0.680000,3160.250000,10.000000,36.389800,8.869950
max,9999.00000,84.000000,14.000000,9.000000,38.000000,1.000000,4998.000000,19.000000,54.312000,17.318200


**Insight: (Dataset 1)**

**Missing Value & Data Duplikat**
- Tidak ada nilai missing value dan duplikat, sehingga tidak diperlukan cleaning data untuk missing value dan duplikat.

**Tipe Data**
- Terdapat 9 variabel dengan tipe data numerik dan 5 variabel ddengan tipe data kategori dengan total data sebanyak 10.000 data

In [12]:
# ASSESSING: Dataset 1 - Nilai Unik Kolom Kategorikal
print('\n- Nilai Unik Kolom Kategorikal:')
cat_cols_1 = df_lifestyle.select_dtypes(include='object').columns
for col in cat_cols_1:
    print(f'\n🔹 {col} ({df_lifestyle[col].nunique()} unique):')
    print(df_lifestyle[col].value_counts().head(10))


- Nilai Unik Kolom Kategorikal:

🔹 gender (4 unique):
gender
Male                 4557
Female               4457
Non-binary            520
Prefer not to say     466
Name: count, dtype: int64

🔹 employment_status (4 unique):
employment_status
Employed         5868
Student          2043
Self-employed    1045
Unemployed       1044
Name: count, dtype: int64

🔹 work_environment (3 unique):
work_environment
On-site    5044
Remote     3009
Hybrid     1947
Name: count, dtype: int64

🔹 mental_health_history (2 unique):
mental_health_history
No     6969
Yes    3031
Name: count, dtype: int64

🔹 seeks_treatment (2 unique):
seeks_treatment
No     6012
Yes    3988
Name: count, dtype: int64

🔹 mental_health_risk (3 unique):
mental_health_risk
Medium    5892
High      2369
Low       1739
Name: count, dtype: int64


In [14]:
# ASSESSING: Dataset 1 - Range & Outlier Check
print('- Range Nilai Numerik DS1:')
num_cols_1 = df_lifestyle.select_dtypes(include=np.number).columns
for col in num_cols_1:
    mn, mx = df_lifestyle[col].min(), df_lifestyle[col].max()
    q1 = df_lifestyle[col].quantile(0.25)
    q3 = df_lifestyle[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((df_lifestyle[col] < q1 - 1.5*iqr) | (df_lifestyle[col] > q3 + 1.5*iqr)).sum()
    print(f'  {col:40s} Min={mn:.2f}, Max={mx:.2f}, Outliers={outliers}')

- Range Nilai Numerik DS1:
  age                                      Min=18.00, Max=65.00, Outliers=0
  stress_level                             Min=1.00, Max=10.00, Outliers=0
  sleep_hours                              Min=3.00, Max=10.00, Outliers=0
  physical_activity_days                   Min=0.00, Max=7.00, Outliers=0
  depression_score                         Min=0.00, Max=30.00, Outliers=0
  anxiety_score                            Min=0.00, Max=21.00, Outliers=0
  social_support_score                     Min=0.00, Max=100.00, Outliers=0
  productivity_score                       Min=42.80, Max=100.00, Outliers=0


In [ ]:
# VISUALISASI: Boxplot per fitur untuk Outlier Check